# task 5 -> feedback source, rlvr versus rlaif

## imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.data import load_yaml, read_jsonl, repo_path

## global env variables ->

In [ ]:
#papermill overrides CONFIG for the smoke test, the defaults are the real run
CONFIG = 'configs/feedback.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir']) / 'task5_feedback'
FIGURES = RESULTS / 'figures'

POLICIES = ['sft', 'rlvr', 'rlaif'] #sft = the untouched base model, rlvr / rlaif = the two supplied frozen adapters
POLICY_COLORS = {'sft': '#52514e', 'rlvr': '#2a78d6', 'rlaif': '#eb6834'}
VARIANTS = ['corrupt_reasoning_correct_final', 'persuasive_filler_correct', 'good_reasoning_wrong_final', 'gold_distractor_wrong_final']
#what a mechanism did with a clean-vs-perturbed pair -> preferred the better (clean) response, tied, or preferred the worse one
OUTCOME_COLORS = {'better_rate': '#1baf7a', 'tie_rate': '#c3c2b7', 'wrong_rate': '#d03b3b'}

PYTHON = sys.executable #the interpreter running this notebook, so the scripts use the same environment

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True})


def run(command):
    #runs one of the release's entry points as its own process, exactly as it would be typed in a terminal.
    #each script loads its own models and writes its results to disk, nothing is passed through notebook state
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'

## in-domain comparison ->

sft, rlvr and rlaif on the fixed gsm8k evaluation subset, exact verifier and pairwise ai judge

In [ ]:
run(f'{PYTHON} -m task5_feedback.evaluate_math --config {CONFIG} --dataset gsm')

## reward-sensitivity and robustness study ->

the supplied 100-response diagnostic set scored by both feedback mechanisms, no policy involved

In [ ]:
run(f'{PYTHON} -m task5_feedback.score_perturbations --config {CONFIG}')

## out-of-domain comparison ->

the same three policies on the fixed svamp transfer set, no retraining

In [ ]:
run(f'{PYTHON} -m task5_feedback.evaluate_math --config {CONFIG} --dataset transfer')

## combined comparison

In [ ]:
run(f'{PYTHON} -m task5_feedback.compare_feedback --config {CONFIG}')

## in-domain results

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
gsm = pd.read_csv(RESULTS / 'summary_gsm.csv').set_index('policy')
display(gsm[['n', 'exact_accuracy', 'format_compliance', 'length_mean', 'length_std', 'truncation_rate',
             'win_rate_vs_sft', 'wins_vs_sft', 'ties_vs_sft', 'losses_vs_sft', 'identical_to_sft']].round(4))

#agreement between the exact verifier and the ai preference signal, per policy pair. 'decided' pairs are those where
#exactly one of the two responses is correct, the only pairs on which the binary verifier expresses a preference
display(pd.read_csv(RESULTS / 'agreement_gsm.csv').set_index('comparison').T.round(4))

## controlled diagnostic results

In [ ]:
rates = pd.read_csv(RESULTS / 'diagnostic_preference_rates.csv')
display(rates.round(3))

#one bar per perturbation type and mechanism, split into prefers-better / tie / prefers-worse (shares add up to 1)
fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), sharey=True)
for axis, mechanism in zip(axes, ['rlvr_exact_verifier', 'rlaif_pairwise_judge']):
    frame = rates[rates['mechanism'] == mechanism].set_index('variant_type').loc[VARIANTS]
    left = np.zeros(len(VARIANTS))
    for column, label in [('better_rate', 'prefers the clean response'), ('tie_rate', 'tie'), ('wrong_rate', 'prefers the perturbed response')]:
        bars = axis.barh([v.replace('_', ' ') for v in VARIANTS], frame[column], left=left, color=OUTCOME_COLORS[column],
                         edgecolor='#fcfcfb', linewidth=2, label=label)
        for bar, share in zip(bars, frame[column]):
            if share >= 0.06:
                axis.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2, f'{share:.2f}',
                          ha='center', va='center', color='#0b0b0b', fontsize=9)
        left += frame[column].to_numpy()
    axis.set_title(mechanism.replace('_', ' '))
    axis.set_xlabel('share of the 20 controlled pairs')
    axis.set_xlim(0, 1)
    axis.grid(axis='y', visible=False)
axes[0].invert_yaxis()
axes[1].legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
fig.savefig(FIGURES / 'diagnostic_preferences.png', dpi=200)
plt.show()

In [ ]:
#S_reason  = Pr[R(clean) > R(reason-corrupt)]      on the pairs with the same correct final answer
#S_outcome = Pr[R(correct final) > R(wrong final)] on the outcome-changing pairs
#a tie is "not greater". the binary verifier ties on every reasoning-only change by design, that invariance is a result
display(pd.read_csv(RESULTS / 'diagnostic_sensitivity.csv').set_index('mechanism').T.round(3))

#the reward each mechanism would give the five variants as one sampled group. the two columns are on different scales
#(binary exact reward against pairwise win rate), only the ordering of the rows inside each column is comparable
group_rewards = pd.read_csv(RESULTS / 'diagnostic_group_rewards.csv')
display(group_rewards.groupby('variant_type')[['rlvr_reward', 'rlaif_reward']].mean().round(3))
print(json.loads((RESULTS / 'diagnostic_verifier_check.json').read_text()))

## out-of-domain results

In [ ]:
transfer = pd.read_csv(RESULTS / 'summary_transfer.csv').set_index('policy')
display(transfer[['n', 'exact_accuracy', 'format_compliance', 'length_mean', 'length_std', 'truncation_rate',
                  'win_rate_vs_sft', 'wins_vs_sft', 'ties_vs_sft', 'losses_vs_sft', 'identical_to_sft']].round(4))
display(pd.read_csv(RESULTS / 'agreement_transfer.csv').set_index('comparison').T.round(4))

#in-domain against transfer for every policy, with the drops
comparison = pd.read_csv(RESULTS / 'feedback_policy_comparison.csv').set_index('policy')
display(comparison.T.round(4))

In [ ]:
#exact accuracy and format compliance, gsm8k next to svamp. one colour per policy, the two sets side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
positions = np.arange(2)
for axis, column, title in zip(axes, ['exact_accuracy', 'format_compliance'], ['exact final-answer accuracy', 'format compliance']):
    for offset, name in zip([-0.26, 0.0, 0.26], POLICIES):
        values = [comparison.loc[name, f'gsm_{column}'], comparison.loc[name, f'transfer_{column}']]
        bars = axis.bar(positions + offset, values, width=0.24, color=POLICY_COLORS[name], label=name)
        axis.bar_label(bars, fmt='%.2f', padding=3, color='#0b0b0b', fontsize=9)
    axis.set_xticks(positions, ['gsm8k (in-domain)', 'svamp (transfer)'])
    axis.set_ylim(0, 1.1)
    axis.set_title(title)
    axis.grid(axis='x', visible=False)
axes[1].legend(frameon=False, loc='upper left', bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
fig.savefig(FIGURES / 'accuracy_in_domain_vs_transfer.png', dpi=200)
plt.show()

In [ ]:
#failure types -> every response is exactly one of correct / wrong final answer / missing final format / truncated
display(pd.read_csv(RESULTS / 'feedback_failure_types.csv').set_index(['dataset', 'policy']).round(4))

#svamp accuracy by problem type, to see which kinds of problem the transfer drop comes from
svamp = pd.concat([pd.DataFrame(read_jsonl(RESULTS / f'generated_transfer_{name}.jsonl')) for name in POLICIES])
display(svamp.pivot_table(index='problem_type', columns='policy', values='exact', aggfunc='mean')[POLICIES]
        .assign(n=svamp[svamp['policy'] == 'sft'].groupby('problem_type').size()).round(3))

## feedback mechanisms side by side

In [ ]:
display(pd.read_csv(RESULTS / 'feedback_mechanism_comparison.csv').set_index('mechanism').T.round(3))

## qualitative candidates ->

shortlists to read through and pick the report examples from, nothing here is selected automatically

In [ ]:
pd.set_option('display.max_colwidth', 600)

#controlled pairs on which the verifier and the judge disagree, grouped the way the manual asks for examples ->
#reasoning-only corruption, persuasive filler, and wrong-final / gold-distractor behaviour
pairs = pd.DataFrame(read_jsonl(RESULTS / 'diagnostic_pairs.jsonl'))
groups = {'reasoning-only corruption': ['corrupt_reasoning_correct_final'],
          'persuasive filler': ['persuasive_filler_correct'],
          'wrong final / gold distractor': ['good_reasoning_wrong_final', 'gold_distractor_wrong_final']}
for title, variants in groups.items():
    frame = pairs[pairs['variant_type'].isin(variants)]
    print(f'\n{title}: verifier {frame["verifier"].value_counts().to_dict()}  judge {frame["judge"].value_counts().to_dict()}')
    disagree = frame[frame['verifier'] != frame['judge']]
    display(disagree[['problem_id', 'variant_type', 'verifier', 'judge', 'question', 'clean_response', 'variant_response']].head(3))

In [ ]:
#the policies' own gsm8k responses where the two signals pull apart -> the judge prefers the response the verifier
#marks wrong, or the judge picks a side although both responses are equally right / wrong for the verifier
pairwise = pd.DataFrame(read_jsonl(RESULTS / 'pairwise_gsm.jsonl'))
responses = {name: pd.DataFrame(read_jsonl(RESULTS / f'generated_gsm_{name}.jsonl')).set_index('problem_id') for name in POLICIES}

def show(rows, title, limit=3):
    print(f'\n{title}: {len(rows)} pairs')
    out = []
    for _, row in rows.head(limit).iterrows():
        a, b = responses[row['first']].loc[row['problem_id']], responses[row['second']].loc[row['problem_id']]
        out.append({'pair': f"{row['first']} (A) vs {row['second']} (B)", 'judge': row['judge_preference'], 'verifier': row['verifier_preference'],
                    'gold': a['gold_final'], 'A_final': a['predicted_final'], 'B_final': b['predicted_final'],
                    'question': a['question'], 'A_response': a['response'], 'B_response': b['response']})
    display(pd.DataFrame(out))

decided = pairwise[pairwise['verifier_preference'] != 'TIE']
show(decided[(decided['judge_preference'] != decided['verifier_preference']) & (decided['judge_preference'] != 'TIE')],
     'judge prefers the response with the wrong final answer')
show(pairwise[(pairwise['verifier_preference'] == 'TIE') & (pairwise['judge_preference'] != 'TIE') & ~pairwise['identical_responses']],
     'judge separates two responses the verifier scores the same')